# Suite JRN — Journals

A journal is mbse-schemas' `Changes.Journal`, as plain data: its transactions, in order, each with its unique `id`, as a store's `follow` gives them, a journal of one transaction at a time. It is data, kept in no store: mbse-journals takes the whole transactions after a position, a transaction's id, joins journals and inserts one after a position, splits one into the one-transaction journals a stream carries and joins them back, and writes, appends and reads it in a file of JSON lines; a store holding the same schemas applies it (`Changes.apply`). mbse-basis [the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md#decided), [CAP-273: journals in memory and in local files](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-273) and [CAP-274: journals sent and received](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-274).

A store appears only where a journal is given (`store.follow`) and where one is applied (`Changes.apply`), each made by `made` alone, so that the suite runs over any store given in its place: the journals themselves are never a store's.

In [ ]:
import json
import tempfile
from pathlib import Path

from mbse.Schemas.Framework import JSON, Builders, Changes, Plain
from mbse.Schemas.Framework.Errors import DecodeError
from mbse.Journals.Framework import Journals


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Book": S.OfObject, "Person": S.OfObject})
        .spec([
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
        ])]))])


def made(install):
    """A store holding the schemas `install` defines: the one place the suite makes a store."""
    store = Builders.Store()
    store.transaction(install)
    return store


def add(store, *keys):
    """A transaction of `store` adding a person to its book by each key, named by it."""
    with store.transaction():
        book = store.singleton("crm.TheBook")
        for key in keys:
            store.crm.Person(lambda p, key=key: p.book(lambda a: a.insert(lambda e: e.key(key).book(book))).name(key))


def rename(store, key, name):
    """A transaction of `store` naming the person its book holds by `key`."""
    with store.transaction():
        store.crm.Person(store.singleton("crm.TheBook").contacts({"key": key}), lambda p: p.name(name))


def remove(store, key):
    """A transaction of `store` removing the person its book holds by `key`."""
    with store.transaction():
        store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key(key))))


def contacts(store):
    """The book's contacts, each its key and name, by key, as JSON text."""
    with store.transaction():
        return JSON.dumps(sorted([e.key.get(), e.contact.name.get()] for e in store.singleton("crm.TheBook").contacts))


def applied(journal, store=None):
    """`store`, a store holding the crm schemas made where none is given, with `journal` applied to it, each of its
    one-transaction journals as a follow gives them, in order: read as a `Changes.Journal`, a value of the store's, and
    applied in a transaction of its own (`Changes.apply`)."""
    store = store or made(crm)
    for one in Journals.split(journal):
        value = Plain.FromPlain(store)(store.registered("Changes.Journal"), one)
        store.transaction(lambda txn: Changes.apply(txn, value))
    return store


def ids(journal):
    """A journal's positions: its transactions' ids, in order."""
    return [transaction["id"] for transaction in journal]


def same(a, b):
    """Whether two journals, or any plain data, are the same: the same JSON text."""
    return JSON.dumps(a) == JSON.dumps(b)


def refused(error, call):
    """The message of the `error` `call` raises."""
    try:
        call()
    except error as raised:
        return str(raised)
    raise AssertionError("not refused")

<a id="JRN-01"></a>
## JRN-01 · A store's follow, collected, builds a journal: each transaction the store commits, in order, once, as plain data, each with its id, a position; once the follow ends, none is given

In [ ]:
model = made(crm)
journal = []
end = model.follow(Journals.collect(journal))
add(model, "ada", "bob")
add(model, "cy")
rename(model, "ada", "Ada")
remove(model, "bob")
end()
first, second, third, fourth = ids(journal)
assert len(set(ids(journal))) == 4 and all(isinstance(i, str) and i for i in ids(journal))
assert same(JSON.loads(JSON.dumps(journal)), journal)  # plain data
assert JSON.dumps(journal[2]["changes"]) == '[{"property": {"holder": "o0", "name": "name", "mutations": [{"set": {"value": "Ada"}}]}}]'
assert JSON.dumps(journal[3]["changes"]) == '[{"entry": {"holder": {"reference": {"symbol": "o0"}}, "adjacency": "contacts", "verb": "remove", "links": {}, "properties": {"key": "bob"}}}]'
other, given = made(crm), []
other.follow(Journals.collect(given))()  # ended at once
add(other, "zed")
assert given == []

<a id="JRN-02"></a>
## JRN-02 · The whole transactions after a position are a journal, up to and including one where given, from the first where none is; after the last, none; a position the journal lacks is refused, and the journal sliced is unchanged

In [ ]:
assert ids(Journals.after(journal, first)) == [second, third, fourth]
assert ids(Journals.after(journal, first, third)) == [second, third]
assert ids(Journals.after(journal, None, second)) == [first, second]
assert same(Journals.after(journal), journal) and Journals.after(journal, fourth) == []
assert refused(LookupError, lambda: Journals.after(journal, "t0")) == 'the journal holds no transaction "t0"'
assert refused(LookupError, lambda: Journals.after(journal, third, first)) == f"the journal holds no transaction {JSON.dumps(first)}"
assert ids(journal) == [first, second, third, fourth]

<a id="JRN-03"></a>
## JRN-03 · Journals joined are one journal of their transactions, in order; a transaction arriving again is held once, in its place, the later arrival replacing it; a journal inserted after a position, or before the first, is read in its place; a position the journal lacks is refused

In [ ]:
assert same(Journals.join(journal[:2], journal[2:]), journal) and Journals.join() == []
again = {**journal[1], "changes": []}  # the second, arriving again, other than it was
joined = Journals.join(journal[:3], [again], journal[2:])
assert ids(joined) == ids(journal) and joined[1]["changes"] == [] and journal[1]["changes"] != []
assert same(Journals.insert(journal[:1] + journal[2:], first, journal[1:2]), journal)
assert same(Journals.insert(journal[1:], None, journal[:1]), journal)
assert same(Journals.insert(journal, fourth, []), journal)
assert ids(Journals.insert(journal, first, journal[3:])) == [first, fourth, second, third]  # moved: held once
assert refused(LookupError, lambda: Journals.insert(journal, "t0", journal)) == 'the journal holds no transaction "t0"'

<a id="JRN-04"></a>
## JRN-04 · A journal split into its one-transaction journals, as a follow gives them, each sent as JSON text, is rebuilt by joining them; a peer holding the journal up to a position, sent the slice after an earlier one, catches up, holding each transaction once

In [ ]:
sent = [JSON.dumps(one) for one in Journals.split(journal)]
assert len(sent) == 4 and all(ids(JSON.loads(text)) == [i] for text, i in zip(sent, ids(journal)))
assert same(Journals.join(*(JSON.loads(text) for text in sent)), journal)
peer = journal[:2]  # the peer holds the journal up to the second
receive = Journals.collect(peer)
for one in Journals.split(Journals.after(journal, first)):  # sent from the first: the second arrives again
    receive(JSON.loads(JSON.dumps(one)))
assert same(peer, journal)

<a id="JRN-05"></a>
## JRN-05 · A journal written to a file is one line for each transaction, its JSON text; whole transactions are appended to it, or to one made where there is none; read back, it is the journal, or the part after a position, a transaction appended again read once; a last line not ended is read, and one cut short is refused

In [ ]:
folder = Path(tempfile.mkdtemp())
path, made_by_append, cut = folder / "crm.jsonl", folder / "new.jsonl", folder / "cut.jsonl"
Journals.write(path, journal[:2])
assert path.read_text(encoding="utf-8") == JSON.dumps(journal[0]) + "\n" + JSON.dumps(journal[1]) + "\n"
Journals.append(path, journal[2:])
assert same(Journals.read(path), journal) and same(Journals.read(path, second), journal[2:])
Journals.append(path, journal[3:])  # the last, again
assert same(Journals.read(path), journal) and len(path.read_text(encoding="utf-8").split("\n")) == 6
Journals.append(made_by_append, journal[:1])
assert same(Journals.read(made_by_append), journal[:1])
Journals.write(made_by_append, [])
assert made_by_append.read_text(encoding="utf-8") == "" and Journals.read(made_by_append) == []
cut.write_text(JSON.dumps(journal[0]), encoding="utf-8")  # no line end
assert same(Journals.read(cut), journal[:1])
cut.write_text(JSON.dumps(journal[0]) + "\n" + JSON.dumps(journal[1])[:20], encoding="utf-8")  # a write cut short
assert refused(DecodeError, lambda: Journals.read(cut)) != ""
assert refused(LookupError, lambda: Journals.read(path, "t0")) == 'the journal holds no transaction "t0"'

<a id="JRN-06"></a>
## JRN-06 · A journal, as plain data or read from a file, applied to a store holding the same schemas (`Changes.apply`), makes it hold what the followed store does, and applied again changes nothing; the slice after a position, applied to a store holding the journal up to it, makes it the same

In [ ]:
replica = applied(journal)
assert contacts(replica) == contacts(model) == '[["ada", "Ada"], ["cy", "cy"]]'
applied(Journals.read(path), replica)
assert contacts(replica) == contacts(model)
behind = applied(Journals.after(journal, None, second))
assert contacts(behind) == '[["ada", "ada"], ["bob", "bob"], ["cy", "cy"]]'
applied(Journals.after(journal, second), behind)
assert contacts(behind) == contacts(model)

<a id="JRN-07"></a>
## JRN-07 · What is not a journal as plain data, a list of its transactions each with its id, is refused by each function, and nothing is written

In [ ]:
bad_journals = [{"id": "t1"}, [{"changes": []}], [{"id": 1}], ["t1"]]
message = "a journal, as plain data, is a list of its transactions, each with its id"
calls = [Journals.after, lambda bad: Journals.join(journal, bad), lambda bad: Journals.insert(bad, None, journal),
         lambda bad: Journals.insert(journal, None, bad), Journals.split, Journals.collect,
         lambda bad: Journals.collect(list(journal))(bad), lambda bad: Journals.write(folder / "bad.jsonl", bad),
         lambda bad: Journals.append(folder / "bad.jsonl", bad)]
assert all(refused(ValueError, lambda: call(bad)) == message for call in calls for bad in bad_journals)
assert not (folder / "bad.jsonl").exists()
(folder / "bad.jsonl").write_text('"t1"\n', encoding="utf-8")
assert refused(ValueError, lambda: Journals.read(folder / "bad.jsonl")) == message